In [7]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)


DATA_DIR = Path("initial_data")

def suggest_ucb_point(X, y, beta=1.5, n_candidates=10_000, random_state=0):
    """
    Given current data (X, y) for ONE function, suggest the next query point x* using
    a Gaussian Process + Upper Confidence Bound (UCB) heuristic.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        The inputs already tried for this function.
        N = number of past points, d = input dimension (2, 3, 4, ..., 8 here).
    y : np.ndarray, shape (N,)
        The outputs observed for those inputs (same order as rows in X).
    beta : float
        Controls exploration vs exploitation in UCB = mean + beta * std.
        - Larger beta = more exploration (try uncertain points).
        - Smaller beta = more exploitation (stay near current best).
    n_candidates : int
        Base number of random candidate points to try in [0, 1]^d.
        We'll possibly increase this for higher dimensions.
    random_state : int
        Seed for the random number generator so results are reproducible.

    Returns
    -------
    best_x : np.ndarray, shape (d,)
        The chosen next input point (in [0, 1]^d) to submit as query.
    gp : GaussianProcessRegressor
        The fitted GP model, in case you want to inspect predictions later.
    """

    # --------------------------
    # 1. Basic info about X, y
    # --------------------------
    # X has shape (N, d): N = number of rows (past points), d = dimensions
    N, d = X.shape

    # --------------------------
    # 2. Build the GP kernel
    # --------------------------
    # RBF kernel:
    #   - Think of this as encoding the idea that nearby points in input space
    #     should have similar outputs (smoothness).
    #   - length_scale=np.ones(d) means we start by assuming each dimension
    #     has a length scale of 1.0; the GP can adapt this when fitting.
    #
    # WhiteKernel:
    #   - Adds some noise to the diagonal of the covariance matrix.
    #   - This allows the GP to handle noisy observations instead of trying
    #     to pass exactly through every point.
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    # --------------------------
    # 3. Create and fit the GP
    # --------------------------
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,   # Centers/scales y internally, helpful when magnitudes differ
        random_state=random_state,
    )

    # Fit the GP on the current data (this is where it "learns" the function shape)
    gp.fit(X, y)

    # --------------------------
    # 4. Generate candidate points
    # --------------------------
    # We'll sample random points uniformly within [0, 1]^d.
    # For higher dimensions we increase the number of candidates because
    # the space is bigger and more "empty".
    if d <= 4:
        n = n_candidates
    else:
        # Double for d >= 5 to explore a bit more widely
        n = n_candidates * 2

    # modern NumPy RNG – local, reproducible
    rng = np.random.default_rng(random_state)
    # Xcand has shape (n_candidates, d), with each coordinate between 0 and 1
    Xcand = rng.uniform(0.0, 1.0, size=(n_candidates, d))

    # --------------------------
    # 5. GP predictions at candidates
    # --------------------------
    # For each candidate point, we ask the GP for:
    #   - mu: predicted mean (what y value we expect)
    #   - std: predictive standard deviation (how uncertain we are)
    mu, std = gp.predict(Xcand, return_std=True)

    # --------------------------
    # 6. Compute UCB score
    # --------------------------
    # UCB(x) = mu(x) + beta * std(x)
    #   - mu(x): exploitation (high mean is good)
    #   - std(x): exploration (high uncertainty is good)
    # beta trades between them.
    ucb = mu + beta * std

    # I found that we had cases where the GP would suggest points that were
    # very close to points that were already evaluated. This was a problem
    # because the GP would then suggest the same point multiple times in a row.
    # This is a simple fix: avoid re-using any already-evaluated points.
    # Vectorized version: compute, in one NumPy broadcasted operation, which
    # candidate points are (within tolerance) equal to ANY seen point, instead
    # of looping over each seen point in Python. This removes Python-loop
    # overhead, scans the candidates once, and makes the intent ("mask any
    # candidate that matches any seen point") clearer.
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-8),
        axis=2,
    )  # shape: (n_candidates, N)
    mask_any_seen = np.any(same_point, axis=1)  # shape: (n_candidates,)
    ucb[mask_any_seen] = -np.inf

    # --------------------------
    # 7. Choose the best candidate
    # --------------------------
    # Take the candidate with the highest UCB value.
    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]  # shape (d,)

    return best_x, gp


In [8]:
with open("inputs-m12.txt") as f:
    prev_inputs = eval(f.read(), {"np": np, "array": np.array})

with open("outputs-m12.txt") as f:
    prev_outputs = eval(f.read(), {"np": np})

# Sanity: should be 8 entries each
assert len(prev_inputs) == 8, "Expected 8 previous inputs"
assert len(prev_outputs) == 8, "Expected 8 previous outputs"

# ==================================
# Per-function beta values
# ==================================
# Based on last week's performance:
# - Explore more (beta=2.0): functions that did poorly or look flat (1, 3, 7)
# - Keep balance (beta=1.5): functions with OK but not amazing results (2, 5, 6)
# - Exploit more (beta=1.0): functions that gave new bests (4, 8)

beta_map = {
    1: 2.0,  # Function 1: flat/weak → more exploration
    2: 1.5,  # Function 2: decent → keep balance
    3: 2.0,  # Function 3: weak → more exploration
    4: 1.0,  # Function 4: big win → more exploitation
    5: 1.5,  # Function 5: good but huge range → keep balance
    6: 1.5,  # Function 6: okay → keep balance
    7: 2.0,  # Function 7: weak → more exploration
    8: 1.0,  # Function 8: big win → more exploitation
}

for i in range(1, 9):
    # 1. Load initial data for this function
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    # 2. Get last week's query and output for this function
    x_prev = np.array(prev_inputs[i - 1])
    y_prev = float(prev_outputs[i - 1])

    print(x_prev)
    print(y_prev)

    # 3. Append them to form updated dataset
    X = np.vstack([X_init, x_prev.reshape(1, -1)])
    y = np.append(y_init, y_prev)

    # 4. Select beta for this function
    beta = beta_map[i]

    # 5. Suggest new query with GP + UCB on updated data
    best_x, gp = suggest_ucb_point(
        X,
        y,
        beta=beta,
        n_candidates=10_000,
        random_state=40 + i,  # different seed per function, but deterministic
    )

    # 6. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)

    # 7. Optional: show best y so far for context
    best_y_so_far = y.max()

    print(f"Function {i}:")
    print(f"  beta used: {beta}")
    print(f"  best y so far (after adding last week): {best_y_so_far:.6f}")
    print(f"  new query to submit: {query_str}")
    print()

[0.954151 0.767932]
9.14184425020275e-88
Function 1:
  beta used: 2.0
  best y so far (after adding last week): 0.000000
  new query to submit: 0.125971-0.826988

[0.773956 0.438878]
0.33661711576593556
Function 2:
  beta used: 1.5
  best y so far (after adding last week): 0.611205
  new query to submit: 0.858598-0.697368

[0.652299 0.043775 0.02003 ]
-0.14950216326847035
Function 3:
  beta used: 2.0
  best y so far (after adding last week): -0.034835
  new query to submit: 0.839213-0.587143-0.224705

[0.372882 0.447822 0.347304 0.45138 ]
-0.25635124958376876
Function 4:
  beta used: 1.0
  best y so far (after adding last week): -0.256351
  new query to submit: 0.411939-0.431631-0.267425-0.398181

[0.573131 0.528491 0.76365  0.811693]
179.70460992197386
Function 5:
  beta used: 1.5
  best y so far (after adding last week): 1088.859618
  new query to submit: 0.510229-0.779340-0.796113-0.594755

[0.017255 0.359724 0.834147 0.516449 0.535116]
-1.28830508383951
Function 6:
  beta used: 1.5